In [39]:
from dotenv import load_dotenv
load_dotenv()
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
import datetime
from langchain_openai import ChatOpenAI
from typing import Annotated,List,TypedDict
from langgraph.graph import add_messages, StateGraph, END,MessagesState
from langgraph.prebuilt import ToolNode
import os
from langchain_core.messages import HumanMessage
llm=ChatOpenAI(model="gpt-4o")


In [4]:
class ChildState(TypedDict):
    messages: Annotated[list, add_messages]

In [54]:
search_tool=TavilySearch(max_results=3,)
@tool
def system_time(format:str="%Y-%m-%d %H:%M:%S"):
    """
    This function return current date and time in specified format
    """

    currentime=datetime.datetime.now()
    formatted=currentime.strftime(format)
    return formatted
tools=[search_tool,system_time]

In [55]:
llm_with_tool=llm.bind_tools(tools)

def agent(state:MessagesState):
    return {
        "messages": [llm_with_tool.invoke(state["messages"])]
    }

def tool_router(state:MessagesState):
    last_message=state["messages"][-1]

    if (hasattr(last_message,"tool_calls") and len(last_message.tool_calls)>0):
        return "tool_node"
    return "END"

In [56]:


tool_node=ToolNode(tools=tools)
subgraph=StateGraph(MessagesState)

subgraph.add_node("agent_mode",agent)
subgraph.add_node("tool_node",tool_node)

subgraph.set_entry_point("agent_mode")
subgraph.add_conditional_edges("agent_mode",tool_router,
                               {"END":END,
                                "tool_node":"tool_node"})
subgraph.add_edge("tool_node","agent_mode")
search_app = subgraph.compile()

In [57]:
search_app.invoke({"messages":[HumanMessage(content="what is time now")]})

{'messages': [HumanMessage(content='what is time now', additional_kwargs={}, response_metadata={}, id='d269da29-c17b-4ac8-b239-9ce373889848'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_pkSfl83ZbWVluG6tetzeX3R8', 'function': {'arguments': '{}', 'name': 'system_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 1314, 'total_tokens': 1324, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_1827dd0c55', 'id': 'chatcmpl-CJ1nRFQZMgf5A9jfy7NN0shcqV6EJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--f3b44ce6-4d1e-4160-8ebf-864f4108908b-0', tool_calls=[{'name': 'system_time', 'args': {}, 'id': 'call_pkSfl83ZbWVluG6tetzeX3R8', 'type': 'tool_ca

In [71]:
from typing import TypedDict, Annotated
from langgraph.graph import add_messages, StateGraph, START, END
from langchain_core.messages import HumanMessage

# Define parent graph with the same schema
class ParentState(TypedDict):
    messages: Annotated[list, add_messages]

# Create parent graph
parent_graph = StateGraph(ParentState)

# Add the subgraph as a node
parent_graph.add_node("search_agents", search_app)

# Connect the flow
parent_graph.add_edge(START, "search_agents")
parent_graph.add_edge("search_agents", END)

# Compile parent graph
parent_app = parent_graph.compile()

# Run the parent graph
result = parent_app.invoke({"messages": [HumanMessage(content="How is the weather in Chennai?")]})

In [66]:
for res in result["messages"]:
    print(res)

content='How is the weather in Chennai?' additional_kwargs={} response_metadata={} id='d7b6f5d1-0245-43c3-a434-004eff714076'
content='' additional_kwargs={'tool_calls': [{'id': 'call_5PJAb6YlHXtJRnDhdJwSQ1SV', 'function': {'arguments': '{"query":"current weather in Chennai"}', 'name': 'tavily_search'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 1317, 'total_tokens': 1336, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1280}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_1827dd0c55', 'id': 'chatcmpl-CJ1qKZmZt3ByVgSV2bme5OYrBDO2p', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--568e934e-84fe-4941-bb25-e4e052b95fad-0' tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in Chennai'}, 'id': 'ca

In [75]:
from typing import TypedDict, Annotated, Dict
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage

# Define parent graph with different schema
class QueryState(TypedDict):
    query: str
    response: str


def search_agent(state:QueryState):
    subgraph_input = {
        "messages": [HumanMessage(content=state["query"])]
    }

    subgraph_result = search_app.invoke(subgraph_input)
    ass_msg=subgraph_result["messages"][-1]
    return {"response": ass_msg.content}

# Create parent graph
parent_graph = StateGraph(QueryState)

parent_graph.add_node("search_agent", search_agent)

# Connect the flow
parent_graph.add_edge(START, "search_agent")
parent_graph.add_edge("search_agent", END)

# Compile parent graph
parent_app = parent_graph.compile()

# Run the parent graph
result = parent_app.invoke({"query": "How is the weather in Chennai?", "response": ""})
print(result)
    

{'query': 'How is the weather in Chennai?', 'response': 'The current weather in Chennai is partly cloudy with a temperature of 30.2°C (86.4°F). The wind is blowing from the west at 16.6 kph (10.3 mph), and the humidity level is 79%. The weather feels like 35.7°C (96.2°F). Additionally, there are showers expected in the vicinity, with temperatures ranging from a minimum of 25°C to a maximum of 34°C today.'}


Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


KeyError: 'query'